In [28]:
import random
import os
import pandas as pd

from sklearn.model_selection import KFold, StratifiedKFold

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, BertModel, BertConfig, AutoConfig
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,f1_score, classification_report,recall_score, precision_score
import numpy as np
from sklearn.model_selection import train_test_split


In [29]:
#CONFIG

#General 
CATEG = "pc" #Category name in dataset
MODEL_OUT_DIR = 'model/' #Path to save model and files
TEST_DATA  = 'test_total_stratified v2.csv' #Path Test Data
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased' #Model name


#Device 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
MAX_LEN = 122


In [30]:
dados_teste = pd.read_csv(TEST_DATA)


In [31]:
seed_val = 22
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,max_lenght=MAX_LEN)

class GerarDataset(Dataset):
  def __init__(self, data, tokenizer):
        self.df = data.reset_index()
        self.tokenizer = tokenizer

  def __len__(self):
        return self.df.shape[0]

  def __getitem__(self, index):
        #Select the sentence and label at the specified index in the data frame
        texto_tratado = self.df.loc[index, 'text']
        if pd.isna(texto_tratado):
          texto_tratado = ""    
        target = self.df.loc[index, CATEG]
        #identifier = self.df.loc[index, 'id']
        tokens = tokenizer(texto_tratado,
                           padding='max_length',
                           max_length=MAX_LEN,
                           add_special_tokens=True,
                           return_tensors='pt',
                           truncation=True)
        
        input_ids = tokens["input_ids"].clone().detach()
        #torch.tensor(tokens["input_ids"]) codigo antigo - warning 
        attention_mask = tokens["attention_mask"].clone().detach()
        target = torch.tensor(target, dtype=torch.long)
        #torch.tensor(target, dtype=torch.long)
        return input_ids, attention_mask, target


model_preds_list = []


In [32]:
def TestarModelo(nome_modelo_teste):
    
    caminho_modelo = os.path.join("model",nome_modelo)

    #Modelo 
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

    # Tokenizando e transformando em inputs (att masks, labels, etc.)
    teste_data = GerarDataset(dados_teste,tokenizer)
    teste_loader = DataLoader(teste_data, batch_size=BATCH_SIZE, shuffle=True)

    model.load_state_dict(torch.load(caminho_modelo))
    model.to(DEVICE)
    model.eval()
    y_real, y_pred = [], []  # Armazenam as previsões e os rótulos reais
    with torch.no_grad():
        for i, (input_ids, attention_mask, target) in enumerate(iterable=teste_loader):
            input_ids, attention_mask, target = input_ids.to(DEVICE), attention_mask.to(DEVICE), target.to(DEVICE)
            #labels = []
            #print(f"input_ids shape: {input_ids.shape}, attention_mask shape: {attention_mask.shape}")
            # Classificação
            input_ids = input_ids.squeeze(1)  # Remove a dimensão extra
            attention_mask = attention_mask.squeeze(1)  # Remove a dimensão extra
            
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            #val_loss = criterion(output.logits,target)
            #val_loss_total += val_loss.item()  # Soma a perda para calcular a média depois
            
            # Previsões
            preditos = torch.argmax(output.logits, 1).to("cpu").tolist()
            y_pred.extend(preditos)  # Armazena as previsões
            y_real.extend(target.to("cpu").tolist()) # Armazena os rótulos reais
      # Converte as listas para numpy arrays para calcular as métricas
    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    # Calcula as principais métricas de classificação
    f1 = f1_score(y_real, y_pred, average='macro',zero_division=0)
    acc = accuracy_score(y_real, y_pred)
    precision = precision_score(y_real, y_pred, average='macro',zero_division=0)
    recall = recall_score(y_real, y_pred, average='macro',zero_division=0)

    f1_class1 = f1_score(y_real, y_pred, average='binary',zero_division=0)
    precision_class1 = precision_score(y_real, y_pred, average='binary',zero_division=0)
    recall_class1 = recall_score(y_real, y_pred, average='binary',zero_division=0)
        # Exibe um relatório de classificação detalhado
    print("*** Validação ***")
    #print(classification_report(y_real, y_pred))
    # Retorna F1, acurácia, precisão, recall e a perda de validação média
    
    return f1, acc, precision, recall, f1_class1, precision_class1, recall_class1

In [33]:
vals_f1, acc_validacao, precisao_validacao, recall_validacao = [], [], [], []
f1s_class1, precisions_class1, recalls_class1,nomes_modelos = [], [], [],[]
resultados = []
modelos = [i for i in  os.listdir("model/") if i.endswith('bin') and CATEG.lower() in i]
for nome_modelo in modelos:

    f1, acc, precision, recall, f1_class1, precision_class1, recall_class1 = TestarModelo(nome_modelo)  # Substitua por sua função
    
    # Adicione os resultados às listas
    nomes_modelos.append(nome_modelo)
    vals_f1.append(f1)
    acc_validacao.append(acc)
    precisao_validacao.append(precision)
    recall_validacao.append(recall)
    f1s_class1.append(f1_class1)
    precisions_class1.append(precision_class1)
    recalls_class1.append(recall_class1)

resultados = pd.DataFrame({
    "Model name": nomes_modelos,
    "Test F1": vals_f1,
    "Test Accuracy": acc_validacao,
    "Test Precision": precisao_validacao,
    "Test Recall": recall_validacao,
    "Test F1 (Class 1)": f1s_class1,
    "Test Precision (Class 1)": precisions_class1,
    "Test Recall (Class 1)": recalls_class1
})


In [34]:
resultados.to_excel(f"Base_teste_{CATEG}_v3.xlsx")